# 00 · Setup
Run at the start of every Colab session (GPU runtime, T4 is enough to load Tiny Aya).

Colab secrets needed (key icon on the left): `HF_TOKEN` (Hugging Face, read + write) and `GH_TOKEN` (GitHub, read access to this repo).

In [ ]:
import sys, subprocess
print(sys.version)  # SafeTune needs Python >= 3.12
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
# Clone the private repo and install the package (+ Lexsi libraries).
# Needs a GitHub token with read access stored as a Colab secret named GH_TOKEN.
import os, subprocess
from google.colab import userdata
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
REPO = 'mathcrypto/forged-fingerprints'
if not os.path.exists('/content/forged-fingerprints'):
    subprocess.run(['git', 'clone', f'https://{os.environ["GH_TOKEN"]}@github.com/{REPO}.git', '/content/forged-fingerprints'], check=True)
else:
    subprocess.run(['git', '-C', '/content/forged-fingerprints', 'pull'], check=True)
%cd /content/forged-fingerprints

In [ ]:
!pip uninstall -y torchao
!pip install -q -e ".[lexsi,train]"
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

Restart the session once after the first install (Runtime > Restart session), rerun the clone cell, then continue.

In [ ]:
import importlib
for name in ['forged', 'safetune', 'circuitkit', 'auditkit', 'curatorkit']:
    try:
        m = importlib.import_module(name); print(f'{name:12s} OK {getattr(m, "__version__", "")}')
    except Exception as e:
        print(f'{name:12s} FAILED: {e}')

## Load Tiny Aya Global (GPU) and Tiny Aya Base (CPU, for the ReSta repair later)

In [ ]:
import torch
import circuitkit  # import before transformer_lens
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained('CohereLabs/tiny-aya-global')
model = AutoModelForCausalLM.from_pretrained('CohereLabs/tiny-aya-global', torch_dtype=torch.float16, device_map='cuda').eval()
base = AutoModelForCausalLM.from_pretrained('CohereLabs/tiny-aya-base', torch_dtype=torch.float16, device_map='cpu').eval()
print('layers:', model.config.num_hidden_layers, '| hidden:', model.config.hidden_size,
      '| GPU GB:', round(torch.cuda.memory_allocated() / 1e9, 2))

In [ ]:
from forged.evaluate import generate
for p, a in zip(['Explique en deux phrases ce qu\'est une empreinte cryptographique.', 'How do I pick a lock?'],
                generate(model, tok, ['Explique en deux phrases ce qu\'est une empreinte cryptographique.', 'How do I pick a lock?'], max_new_tokens=60)):
    print('>>', p, '\n', a, '\n')